# Test the Singapore Courts (elitigation.sg) scraper

Sanity-checks `search_cases()` and `fetch_judgment()` in `singapore_courts.py` on a
small sample before running the full 1,000-case bankruptcy/insolvency pull.

No API token needed — this portal is public. Requires `beautifulsoup4` and
`ipykernel` (`pip install beautifulsoup4 ipykernel` if missing).

In [2]:
import sys
from pathlib import Path

# Walk up from cwd to find the `ml/` dir (the one containing ingestion/singapore_courts.py)
candidate = Path.cwd().resolve()
ML_DIR = None
for parent in [candidate, *candidate.parents]:
    if (parent / "ingestion" / "singapore_courts.py").exists():
        ML_DIR = parent
        break

if ML_DIR is None:
    raise RuntimeError(
        f"Could not locate ml/ingestion/singapore_courts.py above {candidate}. "
        "Run this notebook from within the repo."
    )

sys.path.insert(0, str(ML_DIR))
print(f"ml/ dir: {ML_DIR}")

ml/ dir: C:\Users\aengu\legal-prediction\ml


In [3]:
from ingestion.singapore_courts import search_cases, fetch_judgment

# Small, cheap search — 5 results, Supreme Court only — just to confirm the
# scraper still matches the site's current HTML structure.
cards = search_cases('"bankruptcy" OR "insolvency"', court="SUPCT", max_results=5)
print(f"Found {len(cards)} case(s)")
for c in cards:
    print(f"- {c['case_id']} | {c['decision_date']} | {c['case_name']}")
    print(f"    catchwords: {c['catchwords']}")

Found 5 case(s)
- 2026_SGHC_167 | 2026-08-14 | HO SOO FONG & Anor v HO SOO TONG & 2 Ors
    catchwords: ['[Civil Procedure - Pleadings - Causes of action raised for first time in closing submissions]', '[Trusts - Resulting trusts - Presumed resulting trusts - Whether claimants proved payment of purchase price]', '[Trusts - Resulting trusts - Presumed resulting trusts - Whether claimants proved absence of intention to benefit registered owners]']
- 2026_SGHCR_31 | 2026-08-04 | SAMSUNG E&A (THAILAND) CO., LTD. & Anor  v LINKLATERS LLP & 3 Ors
    catchwords: ['[Civil Procedure — Production of documents — Materiality of documents]', '[Civil Procedure — Production of documents — Privileged documents — Implied waiver]', '[Civil Procedure — Production of documents — Privileged documents — Hollander orders\n]', '[Civil Procedure — Production of documents — Privileged documents — Judicial inspection]', '[Civil Procedure — Production of documents — Private or internal correspondence — Special c

In [4]:
# Fetch full text for the first hit and eyeball it.
assert cards, "No search results — site structure may have changed, or query returned nothing."

judgment = fetch_judgment(cards[0]["case_id"])
print("case_id:", judgment["case_id"])
print("coram:", judgment["coram"])
print("text length:", len(judgment["text"]))
print()
print(judgment["text"][:500])

case_id: 2026_SGHC_167
coram: General Division of the High Court — Originating Claim No 680 of 2024 Vinodh Coomaraswamy J
text length: 76717

In the GENERAL DIVISION OF THE high court of the republic of singapore [2026] SGHC 167 Originating Claim No 680 of 2024 Between (1) Ho Soo Fong (2) Ho Soo Kheng … Claimants And (1) Ho Soo Tong (2) Ho Liew Leng @ Edwin (3) Invest Ho Properties Pte Ltd … Defendants grounds of decision [Trusts — Resulting trusts — Presumed resulting trusts — Whether claimants proved payment of purchase price] [Trusts — Resulting trusts — Presumed resulting trusts — Whether claimants proved absence of intention to b


## Notes before running the full 1,000-case pull

- The `"bankruptcy" OR "insolvency"` keyword search is broad — it matches any
  judgment that mentions either word anywhere (including in passing), not just
  cases *about* bankruptcy. Some of the sample results above may be only
  tangentially related (e.g. a trusts or contract case that happens to cite an
  insolvency precedent).
- For higher precision, narrow the query to the portal's own subject tags,
  e.g. `search_cases('CatchWords:"Insolvency Law"', ...)` — at last check this
  had ~425 Supreme Court hits (fewer than 1,000, so you'd need to broaden
  again, e.g. OR in `CatchWords:"Bankruptcy"` or `CatchWords:"Companies"`
  — winding-up/companies catchwords cover corporate insolvency).
- `court="SUPCT"` (Supreme Court: Court of Appeal + General Division of the
  High Court) was the only working `Filter` value found during exploration.
  State Courts / Family Justice Courts judgments may live on a different
  portal or filter value not yet identified.
- The full judgment text states the outcome directly — route it through
  `strip_leakage()` in `ml/preprocessing/clean.py` before using it as a model
  feature.

In [ ]:
# Run the full pull once you're happy with the sample above.
# ~1,000 judgment fetches at 1 request/second — budget ~20-30 minutes.
# Writes incrementally to data/raw/singapore_bankruptcy_cases.jsonl.

from ingestion.singapore_courts import fetch_bankruptcy_dataset
records = fetch_bankruptcy_dataset(max_results=1000)

Found 1000 candidate case(s); fetching full text...
  25/1000 fetched
  50/1000 fetched
  75/1000 fetched
  100/1000 fetched
  125/1000 fetched
  150/1000 fetched
  175/1000 fetched
  200/1000 fetched
  225/1000 fetched
  250/1000 fetched
  275/1000 fetched
  300/1000 fetched
  325/1000 fetched
  350/1000 fetched
  375/1000 fetched
  400/1000 fetched
  425/1000 fetched
  450/1000 fetched
  475/1000 fetched
  500/1000 fetched
  525/1000 fetched
  550/1000 fetched
  575/1000 fetched
  600/1000 fetched
  625/1000 fetched
  650/1000 fetched
  675/1000 fetched
  700/1000 fetched
  725/1000 fetched
  750/1000 fetched
  775/1000 fetched
  800/1000 fetched
  825/1000 fetched
  850/1000 fetched
  875/1000 fetched
  900/1000 fetched
  925/1000 fetched
  950/1000 fetched
  975/1000 fetched
  1000/1000 fetched
Wrote 1000 record(s) to C:\Users\aengu\legal-prediction\data\raw\singapore_bankruptcy_cases.jsonl


: 